# Virtual try-on wardrobe — upload a person + a few garments → an array of views
Upload **one photo of a person** and **a few clothing images**; get a **garment × view** lookbook. Two stages, two
specialists (the [flux-recipes](https://github.com/remyxai/flux-recipes) pattern):

1. **Views** — CatVTON dresses a person at their *existing* pose (it can't re-pose), so we first generate a few
   **face-locked pose variations** of the uploaded person with **PuLID identity** (`identity_story`) — front / angled
   / turned, on a clean background. View 0 is the original upload.
2. **Try-on** — for every **garment × view**, **CatVTON** (`remyxai/catvton-flux-modular`) inpaints only the clothing
   region (auto agnostic mask), so the face is preserved and the garment is worn.

Honest notes: the generated views are a *recognizable-but-synthetic* likeness (PuLID ArcFace ~0.75, not a pixel
clone); each view needs a clear, unoccluded torso for the auto-mask (a held object → weak transfer). OPEN-WEIGHT
(PuLID + FLUX). 80GB A100 recommended (two FLUX bases, staged).

In [ ]:
import subprocess, os
for _ in range(3):
    if subprocess.call(["pip","install","-q","git+https://github.com/huggingface/diffusers.git"])==0: break
!pip install -q transformers accelerate sentencepiece protobuf hf_transfer scikit-image
!pip install -q insightface facexlib onnxruntime-gpu timm einops ftfy opencv-python-headless peft
subprocess.run(["rm","-rf","flux-recipes"])
subprocess.run(["git","clone","-q","-b","main","https://github.com/remyxai/flux-recipes.git"])

In [ ]:
import sys, torch, numpy as np, cv2, gc, io
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
sys.path.insert(0,"flux-recipes")
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
assert torch.cuda.is_available(); print("GPU:",torch.cuda.get_device_name(0))
from PIL import Image, ImageDraw
from IPython.display import display
# ArcFace (identity kept through the try-on) — independent of PuLID, buffalo_l auto-downloads
from insightface.app import FaceAnalysis
_fa=FaceAnalysis(providers=["CPUExecutionProvider"]); _fa.prepare(ctx_id=0, det_size=(640,640))
def arc_emb(pil):
    fi=_fa.get(cv2.cvtColor(np.asarray(pil.convert("RGB")),cv2.COLOR_RGB2BGR))
    if not fi: return None
    fi=sorted(fi,key=lambda x:(x['bbox'][2]-x['bbox'][0])*(x['bbox'][3]-x['bbox'][1]))[-1]
    e=fi['embedding']; return e/(np.linalg.norm(e)+1e-9)
def fmt(x): return f"{x:.2f}" if x is not None else "no-face"
def matrix(rows, row_labels, col_labels, cell=240, pad=6):
    R,C=len(rows),len(rows[0]); W=(C+1)*(cell+pad)+pad; H=(R+1)*(cell+pad)+pad
    g=Image.new("RGB",(W,H),"white"); d=ImageDraw.Draw(g)
    for j,cl in enumerate(col_labels):
        x=(j+1)*(cell+pad)+pad; d.text((x+4,pad+cell//2),str(cl)[:30],fill="black")
    for i in range(R):
        y=(i+1)*(cell+pad)+pad; d.text((pad+4,y+cell//2),str(row_labels[i])[:16],fill="black")
        for j in range(C):
            x=(j+1)*(cell+pad)+pad
            if rows[i][j] is not None: g.paste(rows[i][j].convert("RGB").resize((cell,cell)),(x,y))
    display(g)
print("setup + ArcFace ready")

## Config — how many synthetic views, and their poses (clean, unoccluded torsos)

In [ ]:
N_VIEWS = 2   # synthetic face-locked poses to add BESIDES the uploaded photo (0 = try-on on the upload only)
VIEW_PROMPTS = [
    "a full-body studio photo of the person standing and facing the camera, arms relaxed at sides, plain light-grey background",
    "a full-body studio photo of the person standing at a three-quarter angle, arms at sides, plain light-grey background",
    "a full-body studio photo of the person standing with weight on one leg, arms at sides, plain light-grey background",
][:N_VIEWS]
MAX_GARMENTS = 4
print("views:", 1+N_VIEWS, "(1 uploaded +", N_VIEWS, "generated) | garments cap:", MAX_GARMENTS)

## Upload — a PERSON photo, then a few GARMENT images (Colab). Falls back to generated images headless.

In [ ]:
PERSON=None; GARMENTS=[]
try:
    from google.colab import files
    print(">> Upload ONE person photo:")
    up=files.upload();  PERSON=Image.open(io.BytesIO(next(iter(up.values())))).convert("RGB") if up else None
    print(">> Upload a few GARMENT images (select several):")
    ug=files.upload();  GARMENTS=[Image.open(io.BytesIO(v)).convert("RGB") for v in ug.values()]
except Exception as e:
    print("no interactive upload (", e, ") — will generate fallbacks")
GARMENTS=GARMENTS[:MAX_GARMENTS]
NEED_GEN = (PERSON is None) or (len(GARMENTS)==0)
print("uploaded person:", PERSON is not None, "| garments:", len(GARMENTS), "| need fallback gen:", NEED_GEN)

## Stage 1 — generate the views (FLUX.1-dev): fallbacks if needed + face-locked poses via PuLID

In [ ]:
runner=None
def _runner():
    global runner
    if runner is None:
        from flux_modular import RecipeRunner; runner=RecipeRunner(steps=20)
    return runner
BASE={"name":"base","run":"default","inputs":["prompt"],"params":{"guidance":3.5}}
# fallbacks so the notebook runs end-to-end without an upload
if PERSON is None:
    PERSON=_runner().run(BASE, {"prompt":"a full-body studio photo of a woman with shoulder-length dark hair, standing, arms at sides, plain grey background"}, seed=3)
if len(GARMENTS)==0:
    for p,s in [("a red midi dress, product photo on white",4),("a blue denim jacket, product photo on white",5),("a cream cable-knit sweater, product photo on white",6)]:
        GARMENTS.append(_runner().run(BASE, {"prompt":p}, seed=s))
# synthetic face-locked views from the person photo
VIEWS=[PERSON]; VIEW_LABELS=["upload"]
if N_VIEWS>0:
    C1={"name":"identity_story","run":"batch","inputs":["id_image","scene_prompts"],"params":{"id_weight":1.0,"guidance":3.5}}
    gv=_runner().run(C1, {"id_image":PERSON,"scene_prompts":VIEW_PROMPTS}, seed=0)
    VIEWS+=list(gv); VIEW_LABELS+=[f"view{i+1}" for i in range(len(gv))]
_ref=arc_emb(PERSON)
def arc(im):
    e=arc_emb(im); return float(np.dot(e,_ref)) if (e is not None and _ref is not None) else None
matrix([[v for v in VIEWS]], ["person"], VIEW_LABELS, cell=220)
matrix([[g for g in GARMENTS]], ["garments"], [f"g{i+1}" for i in range(len(GARMENTS))], cell=220)
print("views:", VIEW_LABELS, "| garments:", len(GARMENTS), "| identity(views vs upload):", [fmt(arc(v)) for v in VIEWS])

## Free the generator, load CatVTON (FLUX.1-Fill)

In [ ]:
if runner is not None:
    del runner; runner=None
gc.collect(); torch.cuda.empty_cache()
print("freed FLUX.1-dev; mem:", round(torch.cuda.memory_allocated()/1e9,1),"GB")
from diffusers import ModularPipeline
vton = ModularPipeline.from_pretrained("remyxai/catvton-flux-modular", trust_remote_code=True)
vton.load_components(dtype=torch.bfloat16); vton.to("cuda")
def dress(person, garment):
    g=torch.Generator("cuda").manual_seed(0)
    return vton(person_image=person, garment_image=garment, height=768, width=576,
                guidance_scale=30.0, num_inference_steps=30, generator=g).images[0]
print("CatVTON loaded")

## Stage 2 — try-on: every garment × every view → the wardrobe matrix

In [ ]:
# rows = garments, cols = views; each cell = that garment worn in that view
grid=[]; labels=[]
for gi,gar in enumerate(GARMENTS):
    row=[dress(v, gar) for v in VIEWS]
    grid.append(row); labels.append(f"g{gi+1}")
    print(f"garment {gi+1}/{len(GARMENTS)} dressed across {len(VIEWS)} views")
matrix(grid, labels, VIEW_LABELS, cell=240)
print()
ids=[[fmt(arc(c)) for c in row] for row in grid]
print("identity kept through try-on (ArcFace vs the person), per cell:")
for gi,row in enumerate(ids): print(f"  g{gi+1}:", row)
print("each row = one garment across the views; face preserved (CatVTON edits only the torso). Weak/odd cell = that")
print("view's torso was occluded/ambiguous for the auto-mask — reshoot that view or pass a manual mask.")

## Save the lookbook (zip of every try-on) — download in Colab

In [ ]:
import os, zipfile
os.makedirs("wardrobe", exist_ok=True)
for gi,row in enumerate(grid):
    for vi,cell in enumerate(row):
        cell.save(f"wardrobe/g{gi+1}_{VIEW_LABELS[vi]}.png")
with zipfile.ZipFile("wardrobe.zip","w") as z:
    for f in os.listdir("wardrobe"): z.write(f"wardrobe/{f}")
print("saved", len(grid)*len(VIEWS), "images to wardrobe.zip")
try:
    from google.colab import files; files.download("wardrobe.zip")
except Exception as e:
    print("(headless — wardrobe.zip written to disk)", e)